# Test des fonctions backend sur le dataset Parcoursup

Ce notebook vise à tester et illustrer l’utilisation des fonctions du backend (`src/backend/processing.py`) pour charger, nettoyer et filtrer le dataset Parcoursup.

## Objectif

- Charger le fichier brut depuis `data/raw/`
- Appliquer les fonctions de traitement du backend
- Inspecter rapidement le résultat (colonnes, nombre de lignes)

In [ ]:
import os

os.getcwd()

In [ ]:
# Vérifier que Python cherche et pointe vers les bons dossiers
import sys

sys.path

## 1. Imports et configuration

In [5]:
import pandas as pd
import numpy as np
import re
import pathlib
from pathlib import Path

from src.backend.processing import (
    load_parcoursup_csv,
    drop_unused_columns,
    rename_columns,
    filter_target_year,
)

## 2. Chargement du dataset brut

In [12]:
csv_path = Path("../data/raw/fr-esr-cartographie_formations_parcoursup.csv")
df_raw = load_parcoursup_csv(csv_path)
df_raw.head()

,Session,Identifiant de l'établissement,Nom de l'établissement,Types d'établissement,Types de formation,Nom long de la formation,Mentions/Spécialités,Formations en apprentissage,Internat,Aménagement,...,Lien vers les données statistiques pour l'année antérieure,Site internet de l'établissement,Localisation,Nom court de la formation,Code interne Parcoursup de la formation,Code interne Parcoursup pour les portails,etablissement_id_paysage,composante_id_paysage,rnd,code_formation
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://esirem.u-bourgogne.fr,"47.3122, 5.07398",FI-BAC5,10,7,NaN,NaN,7973,210
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://www.polytech-angers.fr,"47.4805, -0.59213",FI-BAC5,19,7,5Zmt0,NaN,7231,210
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://eeigm.univ-lorraine.fr/,"48.6954, 6.19324",FI-BAC5,21,7,t6Cq5,9pRyR,9655,210
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,https://ensgsi.univ-lorraine.fr/,"48.6949, 6.19356",FI-BAC5,22,7,t6Cq5,Ovw2G,6860,210
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,https://www.imt-nord-europe.fr,"50.61139, 3.13495",FI-BAC5,24,7,NaN,NaN,21437,210


## 3. Nettoyage et préparation

In [13]:
df_clean = (
    df_raw
    .pipe(drop_unused_columns)
    .pipe(rename_columns)
    .pipe(filter_target_year)
)

## 4. Vérifications rapides

In [17]:
df_clean.info()
df_clean["session"].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 24058 entries, 0 to 24057
Data columns (total 19 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   session                                    24058 non-null  object
 1   id_etablissement                           24058 non-null  object
 2   name_etablissement                         24058 non-null  object
 3   type_etablissement                         24058 non-null  object
 4   type_formation                             24058 non-null  object
 5   name_formation                             24058 non-null  object
 6   mentions_specialites                       24058 non-null  object
 7   apprentissage                              9599 non-null   object
 8   internat                                   772 non-null    object
 9   amenagement                                24058 non-null  object
 10  Informations complémentaires           

session
2026    24058
Name: count, dtype: int64

In [18]:
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,Informations complémentaires,region,departement,commune,link_formation,website_etablissement,short_name_formation,Code interne Parcoursup de la formation,Code interne Parcoursup pour les portails
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",Des épreuves de sélection sont mises en place ...,Bourgogne-Franche-Comté,Côte-d'Or,Dijon,https://dossierappel.parcoursup.fr/Candidats/p...,http://esirem.u-bourgogne.fr,FI-BAC5,10,7
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",Des épreuves de sélection sont mises en place ...,Pays de la Loire,Maine-et-Loire,Angers,https://dossierappel.parcoursup.fr/Candidats/p...,http://www.polytech-angers.fr,FI-BAC5,19,7
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",Des épreuves de sélection sont mises en place ...,Grand Est,Meurthe-et-Moselle,Nancy,https://dossierappel.parcoursup.fr/Candidats/p...,http://eeigm.univ-lorraine.fr/,FI-BAC5,21,7
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",Des épreuves de sélection sont mises en place ...,Grand Est,Meurthe-et-Moselle,Nancy,https://dossierappel.parcoursup.fr/Candidats/p...,https://ensgsi.univ-lorraine.fr/,FI-BAC5,22,7
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",Des épreuves de sélection sont mises en place ...,Hauts-de-France,Nord,Villeneuve-d'Ascq,https://dossierappel.parcoursup.fr/Candidats/p...,https://www.imt-nord-europe.fr,FI-BAC5,24,7
